# OSeMOSYS SAND -> LP -> HiGHS -> SOL

Streamlined workflow for `SAND/PD 2TimeSlices 41D_Parameters_SAND 19-05-2026.xlsx`.

This notebook does not write into `/CSV`; generated CSV, LP, SOL, and optional result files go under `resultados/sand_pd_2ts/`.

In [ ]:
from pathlib import Path
import json
import shutil
import sys
import time

REPO_ROOT = Path.cwd().resolve()
BACKEND_DIR = REPO_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

SAND_FILE = REPO_ROOT / "SAND" / "PD 2TimeSlices 41D_Parameters_SAND 19-05-2026.xlsx"
SHEET_NAME = "Parameters"
OUTPUT_DIR = REPO_ROOT / "resultados" / "sand_pd_2ts"
CSV_DIR = OUTPUT_DIR / "csv"
LP_PATH = OUTPUT_DIR / "osemosys_sand_pd_2ts.lp"
SOL_PATH = OUTPUT_DIR / "osemosys_sand_pd_2ts.sol"
RESULTS_DIR = OUTPUT_DIR / "results"

DIV = 1
SOLVER_THREADS = 0  # 0 lets HiGHS choose. Set e.g. 8 to force a thread count.
HIGHSPY_OPTIONS = {
    "presolve": "on",
    # Optional tuning knobs. Leave disabled unless you are comparing numerics.
    # "solver": "simplex",
    # "parallel": "on",
    # "user_bound_scale": -4,
}

print("SAND_FILE:", SAND_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)
assert SAND_FILE.is_file(), f"Missing SAND workbook: {SAND_FILE}"

## 1. Generate Sparse CSV Inputs

The backend exporter handles the cleaned 2-timeslice SAND structure and avoids dense zero-heavy activity-ratio CSVs.

In [ ]:
from app.simulation.core.data_processing import run_data_processing_from_excel

if CSV_DIR.exists():
    shutil.rmtree(CSV_DIR)
CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
processing = run_data_processing_from_excel(
    SAND_FILE,
    CSV_DIR,
    sheet_name=SHEET_NAME,
    div=DIV,
)
print("CSV generation seconds:", round(time.perf_counter() - t0, 2))

set_counts = {
    key: len(values)
    for key, values in processing.sets.items()
    if key in {"REGION", "TECHNOLOGY", "FUEL", "EMISSION", "YEAR", "TIMESLICE", "MODE_OF_OPERATION", "STORAGE", "UDC"}
}
print("Set counts:")
print(json.dumps(set_counts, indent=2))

print("Timeslices:", processing.sets.get("TIMESLICE"))

for name in [
    "TIMESLICE.csv",
    "YearSplit.csv",
    "InputActivityRatio.csv",
    "OutputActivityRatio.csv",
    "EmissionActivityRatio.csv",
    "VariableCost.csv",
]:
    path = CSV_DIR / name
    if path.exists():
        print(f"{name}: {path.stat().st_size / (1024 * 1024):.2f} MB")

if processing.data_quality_warnings:
    print("Data quality warnings:")
    print(json.dumps(processing.data_quality_warnings, indent=2, ensure_ascii=False))

## 2. Build the Pyomo Instance

In [ ]:
from app.simulation.core.instance_builder import build_instance
from app.simulation.core.model_definition import create_abstract_model

t0 = time.perf_counter()
model = create_abstract_model(
    has_storage=processing.has_storage,
    has_udc=processing.has_udc,
)
instance = build_instance(
    model,
    CSV_DIR,
    has_storage=processing.has_storage,
    has_udc=processing.has_udc,
)
print("Instance build seconds:", round(time.perf_counter() - t0, 2))
print("REGION:", len(list(instance.REGION)))
print("YEAR:", len(list(instance.YEAR)))
print("TIMESLICE:", len(list(instance.TIMESLICE)))
print("TECHNOLOGY:", len(list(instance.TECHNOLOGY)))
print("FUEL:", len(list(instance.FUEL)))

## 3. Write LP, Solve with HiGHS, Write SOL

This uses `highspy.readModel()` directly, matching the faster LP-direct pattern from the performance work. If the model is not optimal, later result parsing is skipped instead of failing on missing values.

In [ ]:
import highspy
from app.simulation.core.solver import write_lp_file

t0 = time.perf_counter()
write_lp_file(instance, LP_PATH, symbolic=True)
print("LP write seconds:", round(time.perf_counter() - t0, 2))
print("LP size MB:", round(LP_PATH.stat().st_size / (1024 * 1024), 2))

h = highspy.Highs()
for option, value in HIGHSPY_OPTIONS.items():
    h.setOptionValue(option, value)
if SOLVER_THREADS and SOLVER_THREADS > 0:
    h.setOptionValue("threads", SOLVER_THREADS)

t0 = time.perf_counter()
read_status = h.readModel(str(LP_PATH))
print("readModel status:", read_status)
print("HiGHS read seconds:", round(time.perf_counter() - t0, 2))
print("Rows:", h.getNumRow())
print("Cols:", h.getNumCol())
print("Nonzeros:", h.getNumNz())

t0 = time.perf_counter()
h.run()
print("HiGHS run seconds:", round(time.perf_counter() - t0, 2))

info = h.getInfo()
model_status = h.getModelStatus()
model_status_text = h.modelStatusToString(model_status)
print("Model status:", model_status_text)
print("Objective:", info.objective_function_value)
print("Simplex iterations:", info.simplex_iteration_count)
print("Primal solution status:", h.solutionStatusToString(info.primal_solution_status))
print("Dual solution status:", h.solutionStatusToString(info.dual_solution_status))
print("Basis validity:", h.basisValidityToString(info.basis_validity))

h.writeSolution(str(SOL_PATH), 1)
print("SOL path:", SOL_PATH)
print("SOL size MB:", round(SOL_PATH.stat().st_size / (1024 * 1024), 2))

IS_OPTIMAL = model_status == highspy.HighsModelStatus.kOptimal
if not IS_OPTIMAL:
    print("Skipping solution parsing because model status is not Optimal.")

## 4. Load and Export Results When Optimal

In [ ]:
from app.simulation.core.results_processing import process_results
from app.simulation.core.solver import _load_highs_duals_by_row, _load_highs_solution_by_name
from app.simulation.export_results import export_solution_to_folder

if not IS_OPTIMAL:
    print(f"No result export. HiGHS status: {model_status_text}")
else:
    _load_highs_solution_by_name(instance, h)
    _load_highs_duals_by_row(instance, h)

    sets = processing.sets
    result = process_results(
        instance,
        {
            "solver_name": "highs",
            "solver_status": model_status_text,
            "objective_value": float(info.objective_function_value),
            "solver_threads_used": SOLVER_THREADS or None,
            "reserve_margin_dual": None,
            "infeasibility_diagnostics": None,
        },
        regions=sets.get("REGION", []),
        technologies=sets.get("TECHNOLOGY", []),
        years=sets.get("YEAR", []),
        emissions=sets.get("EMISSION", []),
        has_storage=processing.has_storage,
        region_id_by_name=processing.region_id_by_name,
        technology_id_by_name=processing.technology_id_by_name,
        region_name_by_id=processing.region_name_by_id,
        fuel_id_by_name=processing.fuel_id_by_name,
        emission_id_by_name=processing.emission_id_by_name,
        timeslice_id_by_name=processing.timeslice_id_by_name,
        mode_of_operation_id_by_name=processing.mode_of_operation_id_by_name,
        season_id_by_name=processing.season_id_by_name,
        daytype_id_by_name=processing.daytype_id_by_name,
        dailytimebracket_id_by_name=processing.dailytimebracket_id_by_name,
        storage_id_by_name=processing.storage_id_by_name,
    )
    out = export_solution_to_folder(result, RESULTS_DIR, write_json=True, use_timestamp_subdir=False)
    print("Results exported to:", out)

## 5. Artifact Summary

In [ ]:
for path in [CSV_DIR, LP_PATH, SOL_PATH, RESULTS_DIR]:
    if path.exists():
        if path.is_dir():
            size = sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
            print(f"{path}: directory, {size / (1024 * 1024):.2f} MB")
        else:
            print(f"{path}: file, {path.stat().st_size / (1024 * 1024):.2f} MB")
    else:
        print(f"{path}: not created")